# Final Dataset for Criteria Analysis

Build the final list of works that feed the criteria-extraction pipeline.

Applies the same filters as `11_authors_non_mythic_5_periods.ipynb`:

1. `selected_english_translation == 1`
2. `historian == 0` (non-historian authors only)
3. Exclude `Unknown` and `Pseudo-Plutarch`
4. `author_impact_date` must be present
5. Drop works scored `factuality == 1` (mythic / speculative) from `works_factuality_v18.tsv`
6. Keep only works that fall inside one of the 4 analysis periods

Output: `../data/processed_data/final_dataset_for_criteria.tsv`

In [1]:
from pathlib import Path

import pandas as pd

META_TSV = Path('../data/processed_data/perseus_works_wikidata.tsv')
FACT_TSV = Path('../data/works_factuality_v18.tsv')
OUTPUT_TSV = Path('../data/processed_data/final_dataset_for_criteria.tsv')

In [2]:
EXCLUDE_AUTHORS = {'unknown', 'pseudo-plutarch'}

df = pd.read_csv(META_TSV, sep='\t')
n_start = len(df)

df = df[(df['selected_english_translation'] == 1) & (df['historian'] == 0)].copy()
df = df[~df['perseus_author'].astype(str).str.strip().str.lower().isin(EXCLUDE_AUTHORS)]
df['year'] = pd.to_numeric(df['author_impact_date'], errors='coerce')
df = df[df['year'].notna()].copy()
df['year'] = df['year'].astype(int)
df['n_pages'] = pd.to_numeric(df['n_pages'], errors='coerce').fillna(0).astype(int)

print(f'Start: {n_start} rows')
print(f'After base filters (English, non-historian, named author, impact date): {len(df)}')

Start: 1084 rows
After base filters (English, non-historian, named author, impact date): 391


In [3]:
fact = pd.read_csv(FACT_TSV, sep='\t')[['perseus_id', 'factuality', 'factuality_reason']]
df = df.merge(fact, on='perseus_id', how='left')

before = len(df)
df = df[df['factuality'] != 1].copy()
print(f'After dropping factuality==1 (mythic): {len(df)} (was {before})')

After dropping factuality==1 (mythic): 318 (was 391)


In [4]:
PERIODS = [
    ('Classical (500\u2013360 BCE)',               -500,  -360),
    ('Late Classical (354\u2013165 BCE)',          -354,  -165),
    ('Hellenistic & Early Roman (165 BCE \u2013 105 CE)', -165, 105),
    ('High Roman Empire (135\u2013205 CE)',         135,   205),
]

def assign_period(year):
    for label, lo, hi in PERIODS:
        if lo <= year <= hi:
            return label
    return None

df['period'] = df['year'].apply(assign_period)
before = len(df)
df = df[df['period'].notna()].copy()
print(f'After period filter: {len(df)} (was {before})')
print()
print(df['period'].value_counts())

After period filter: 318 (was 318)

period
Classical (500–360 BCE)                         146
Late Classical (354–165 BCE)                     79
High Roman Empire (135–205 CE)                   58
Hellenistic & Early Roman (165 BCE – 105 CE)     35
Name: count, dtype: int64


In [5]:
summary = (
    df.groupby('period')
      .agg(n_authors=('perseus_author', 'nunique'),
           n_works=('file_id', 'count'),
           total_pages=('n_pages', 'sum'),
           total_words=('n_words', 'sum'))
      .reset_index()
)
summary.loc['TOTAL'] = [
    'TOTAL',
    df['perseus_author'].nunique(),
    len(df),
    int(df['n_pages'].sum()),
    int(df['n_words'].sum()) if 'n_words' in df.columns else 0,
]
summary

,period,n_authors,n_works,total_pages,total_words
0,Classical (500–360 BCE),9,146,4563,1142068
1,Hellenistic & Early Roman (165 BCE – 105 CE),6,35,1385,347221
2,High Roman Empire (135–205 CE),8,58,4920,1230267
3,Late Classical (354–165 BCE),7,79,4956,1238890
TOTAL,TOTAL,30,318,15824,3958446


In [6]:
output_cols = [
    'file_id',
    'perseus_id',
    'perseus_author',
    'perseus_title',
    'wikidata_work_id',
    'wikidata_work_label',
    'author_wikidata_id',
    'author_impact_date',
    'year',
    'period',
    'genre',
    'form_of_creative_work',
    'instance_of',
    'factuality',
    'factuality_reason',
    'main_language',
    'languages',
    'editors',
    'pub_date',
    'n_characters',
    'n_words',
    'n_pages',
    'file_path',
]
output_cols = [c for c in output_cols if c in df.columns]

out = df[output_cols].sort_values(['period', 'year', 'perseus_author', 'perseus_title']).reset_index(drop=True)
OUTPUT_TSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_TSV, sep='\t', index=False)
print(f'Saved {len(out)} works \u2192 {OUTPUT_TSV}')
out.head()

Saved 318 works → ../data/processed_data/final_dataset_for_criteria.tsv


,file_id,perseus_id,perseus_author,perseus_title,wikidata_work_id,wikidata_work_label,author_wikidata_id,author_impact_date,year,period,...,factuality,factuality_reason,main_language,languages,editors,pub_date,n_characters,n_words,n_pages,file_path
0,tlg0027.tlg004.perseus-eng2,tlg0027.tlg004,Andocides,Against Alcibiades,NaN,NaN,Q492228,-500,-500,Classical (500–360 BCE),...,4,"Andocides was a forensic orator, and his works...",English,eng; grc; lat,Kenneth John Maidment,1941,29491,5226,21,tlg0027/tlg004/tlg0027.tlg004.perseus-eng2.xml
1,tlg0027.tlg002.perseus-eng2,tlg0027.tlg002,Andocides,On His Return,Q27663867,On His Return,Q492228,-500,-500,Classical (500–360 BCE),...,4,"Andocides was a forensic orator, and his works...",English,grc; eng; lat,Kenneth John Maidment,1941,16263,2941,12,tlg0027/tlg002/tlg0027.tlg002.perseus-eng2.xml
2,tlg0027.tlg001.perseus-eng2,tlg0027.tlg001,Andocides,On the Mysteries,Q87737021,On the Mysteries,Q492228,-500,-500,Classical (500–360 BCE),...,4,"Andocides was a forensic orator, and his works...",English,grc; eng; lat,Kenneth John Maidment,1941,106628,18619,74,tlg0027/tlg001/tlg0027.tlg001.perseus-eng2.xml
3,tlg0027.tlg003.perseus-eng2,tlg0027.tlg003,Andocides,On the Peace with Sparta,Q87737023,On the Peace with Sparta,Q492228,-500,-500,Classical (500–360 BCE),...,4,"Andocides was a forensic orator, and his works...",English,eng; grc; lat,Kenneth John Maidment,1941,30121,5221,21,tlg0027/tlg003/tlg0027.tlg003.perseus-eng2.xml
4,tlg0028.tlg006.perseus-eng2,tlg0028.tlg006,Antiphon,On the Choreutes,Q87738364,On the Choreutes,Q15078686,-500,-500,Classical (500–360 BCE),...,4,"Antiphon was a forensic orator, and his works ...",English,grc; lat; eng,Kenneth John Maidment,1941,35112,6061,24,tlg0028/tlg006/tlg0028.tlg006.perseus-eng2.xml


In [7]:
author_breakdown = (
    out.groupby(['period', 'perseus_author'])
       .agg(n_works=('file_id', 'count'), total_pages=('n_pages', 'sum'))
       .reset_index()
       .sort_values(['period', 'total_pages'], ascending=[True, False])
)
author_breakdown

,period,perseus_author,n_works,total_pages
8,Classical (500–360 BCE),Plato,33,2121
6,Classical (500–360 BCE),Isocrates,30,819
3,Classical (500–360 BCE),Hippocrates,19,597
7,Classical (500–360 BCE),Lysias,34,349
5,Classical (500–360 BCE),Isaeus,12,208
1,Classical (500–360 BCE),Antiphon,6,135
2,Classical (500–360 BCE),Aristophanes,2,130
0,Classical (500–360 BCE),Andocides,4,128
4,Classical (500–360 BCE),Hyperides,6,76
13,Hellenistic & Early Roman (165 BCE – 105 CE),New Testament,27,699
